In [35]:
import requests
from bs4 import BeautifulSoup
import html2text
import argparse
import re
import sys
from urllib.parse import urlparse


def is_content_element(element):
    if element.name is None:
        return False

    skip_patterns = [
        'header', 'footer', 'nav', 'menu', 'sidebar', 'banner',
        'advertisement', 'cookie', 'popup', 'modal', 'social',
        'comment', 'widget', 'toolbar', 'masthead'
    ]

    attrs = []
    if element.has_attr('class'):
        attrs.extend(element['class'])
    if element.has_attr('id'):
        attrs.append(element['id'])

    attrs = [attr.lower() for attr in attrs]

    for attr in attrs:
        for pattern in skip_patterns:
            if pattern in attr:
                return False

    if element.has_attr('role'):
        role = element['role'].lower()
        skip_roles = ['navigation', 'banner', 'complementary', 'contentinfo']
        if role in skip_roles:
            return False

    return True


def extract_content(soup):
    """
    Extract the main content from BeautifulSoup HTML document.
    Returns a list of content elements (paragraphs, headings, lists, etc.)
    """
    # Dictionary of scores for candidate content containers
    container_scores = {}

    # Identify potential content containers
    containers = soup.find_all(['div', 'article', 'main', 'section'])

    # Score containers based on content density and other heuristics
    for container in containers:
        # Skip if likely to be navigation, sidebar, footer, etc.
        if _is_likely_non_content(container):
            continue

        # Count text content elements
        paragraphs = container.find_all('p')
        headings = container.find_all(['h1', 'h2', 'h3', 'h4', 'h5', 'h6'])
        lists = container.find_all(['ul', 'ol'])

        # Calculate text length
        text_length = len(container.get_text(strip=True))

        # Skip containers with very little text
        if text_length < 200:
            continue

        # Calculate content density (text vs HTML ratio)
        html_length = len(str(container))
        if html_length == 0:
            continue

        density = text_length / html_length

        # Calculate element score based on content indicators
        score = (len(paragraphs) * 5 +
                 len(headings) * 2 +
                 len(lists) * 2 +
                 text_length * 0.1 * density)

        container_scores[container] = score

    # If we found scored containers, use the best one
    if container_scores:
        main_container = max(container_scores.items(), key=lambda x: x[1])[0]

        # Extract all content elements from main container
        content_elements = []
        for element in main_container.find_all(['p', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6',
                                               'ul', 'ol', 'pre', 'blockquote', 'table']):
            if _has_meaningful_content(element):
                content_elements.append(element)

        return content_elements

    # Fallback: if no good container found, extract all paragraphs and headings directly
    content_elements = []
    for element in soup.find_all(['p', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6',
                                 'ul', 'ol', 'pre', 'blockquote', 'table']):
        if _has_meaningful_content(element):
            content_elements.append(element)

    return content_elements

def _is_likely_non_content(element):
    """Check if an element is likely navigation, header, footer, etc."""
    # Check id and class attributes for common non-content indicators
    attributes = []
    if element.get('id'):
        attributes.append(element['id'].lower())
    if element.get('class'):
        attributes.extend([c.lower() for c in element['class']])

    non_content_indicators = [
        'nav', 'navigation', 'menu', 'header', 'footer', 'sidebar',
        'comment', 'advertisement', 'ad', 'banner', 'promo', 'share',
        'social', 'related', 'widget', 'toolbar', 'copyright'
    ]

    for attr in attributes:
        for indicator in non_content_indicators:
            if indicator in attr:
                return True

    return False

def _has_meaningful_content(element):
    """Check if an element has meaningful content (not empty or very short)."""
    text = element.get_text(strip=True)
    # Element should have some text and not be very short (unless it's a heading)
    if not text:
        return False
    if element.name.startswith('h') and len(text) > 3:
        return True
    if len(text) < 20 and element.name == 'p':
        return False
    return True


def scrape_to_markdown(url):
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
            'Accept-Language': 'en-US,en;q=0.5',
            'Referer': 'https://www.google.com/',
            'DNT': '1',
            'Connection': 'keep-alive',
            'Upgrade-Insecure-Requests': '1',
        }

        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, 'html.parser')

        for element in soup(['script', 'style', 'noscript', 'svg', 'iframe']):
            element.decompose()

        content_elements = extract_content(soup)

        if not content_elements:
            return "No content found on the page."

        title = ""
        title_elem = soup.find('title')
        if title_elem:
            title = f"# {title_elem.get_text().strip()}\n\n"

        converter = html2text.HTML2Text()
        converter.ignore_links = False
        converter.ignore_images = False
        converter.ignore_tables = False
        converter.body_width = 0

        markdown_content = title

        for element in content_elements:
            element_html = str(element)
            md_part = converter.handle(element_html)

            md_part = re.sub(r'\n{3,}', '\n\n', md_part)
            markdown_content += md_part + "\n\n"

        markdown_content = re.sub(r'\n{3,}', '\n\n', markdown_content)

        return markdown_content.strip()

    except requests.exceptions.RequestException as e:
        return f"Error: Failed to retrieve the webpage: {str(e)}"
    except Exception as e:
        return f"Error: An unexpected error occurred: {str(e)}"


def get_output_filename(url):
    parsed = urlparse(url)
    domain = parsed.netloc.replace("www.", "")
    path = parsed.path.strip("/").replace("/", "_")
    if not path:
        path = "index"
    return f"{domain}_{path}.md"


In [41]:
url = "http://www.agh.edu.pl/wydarzenia/detail/s/to-bedzie-maj-juwekrk"
output_file = f"{url[8:].replace('.', '-').replace('/', '_')}.md"

markdown_content = scrape_to_markdown(url)

with open(output_file, "w", encoding="utf-8") as f:
    f.write(markdown_content)

print(f"Content saved to {output_file}")

Content saved to ww-agh-edu-pl_wydarzenia_detail_s_to-bedzie-maj-juwekrk.md
